In [11]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import LSTM, Dense
import tensorflow as tf

# Assuming you have a pandas DataFrame with a column named 'load' containing your load data
# Make sure your data is sorted chronologically

# Load your data
# For example:
# df = pd.read_csv('your_data.csv')

# Assuming your data looks like this:
# timestamp            load
# 2023-01-01 00:00:00  100
# 2023-01-01 00:15:00  110
# ...



In [2]:
# write a function to read the data and convert the df to final
def read_data(data_path):
    df = pd.read_csv(data_path)
    final = pd.DataFrame()
    final['ds'] = pd.to_datetime(df['date'])
    final['y'] = df['load_data']
    final['unique_id'] = 'load_data'
    # set the index as ds
    final.index = final['ds']
    return final

In [3]:
path = 'final_output.csv'
df  = read_data(path)

In [4]:
timestamp_s = df.ds.map(pd.Timestamp.timestamp)

In [5]:
day = 24*60*60
year = (365.2425)*day

df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))
df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))

In [6]:
df.head()

,ds,y,unique_id,Day sin,Day cos,Year sin,Year cos
ds,,,,,,,
2022-04-01 00:15:00,2022-04-01 00:15:00,16.07,load_data,0.065403,0.997859,0.999877,0.015657
2022-04-01 00:30:00,2022-04-01 00:30:00,13.75,load_data,0.130526,0.991445,0.999880,0.015478
2022-04-01 00:45:00,2022-04-01 00:45:00,13.13,load_data,0.195090,0.980785,0.999883,0.015299
2022-04-01 01:00:00,2022-04-01 01:00:00,10.36,load_data,0.258819,0.965926,0.999886,0.015120
2022-04-01 01:15:00,2022-04-01 01:15:00,9.96,load_data,0.321439,0.946930,0.999888,0.014941


In [13]:

# read the data


# Normalize the data using MinMaxScaler
scaler = MinMaxScaler()
df['load_scaled'] = scaler.fit_transform(df[['y']])

# Create sequences for training
sequence_length = 20  # You can adjust this based on the context of your data
X, y = [], []

for i in range(len(df) - sequence_length):
    X.append(df['load_scaled'].values[i:i+sequence_length])
    y.append(df['load_scaled'].values[i+sequence_length])

X, y = np.array(X), np.array(y)

# Reshape the data for LSTM input (samples, time steps, features)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

# Split the data into training and testing sets and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Build the LSTM model
model = Sequential()
model.add(LSTM(96, input_shape=(X.shape[1], 1)))
model.add(Dense(48))
model.add(Dense(1))
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mean_squared_error')

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32)

# Evaluate the model
loss = model.evaluate(X_test, y_test)

print(f'Mean Squared Error on Test Data: {loss}')

# Make predictions
predictions = model.predict(X_test)

# Inverse transform the predictions to the original scale
predictions_original = scaler.inverse_transform(predictions)

Epoch 1/10
1241/1241 [==============================] - 18s 12ms/step - loss: 0.0072
Epoch 2/10
1241/1241 [==============================] - 15s 12ms/step - loss: 0.0061
Epoch 3/10
1241/1241 [==============================] - 14s 12ms/step - loss: 0.0061
Epoch 4/10
1241/1241 [==============================] - 16s 13ms/step - loss: 0.0060
Epoch 5/10
1241/1241 [==============================] - 16s 13ms/step - loss: 0.0060
Epoch 6/10
1241/1241 [==============================] - 13s 11ms/step - loss: 0.0060
Epoch 7/10
1241/1241 [==============================] - 13s 11ms/step - loss: 0.0059
Epoch 8/10
1241/1241 [==============================] - 13s 11ms/step - loss: 0.0060
Epoch 9/10
1241/1241 [==============================] - 13s 11ms/step - loss: 0.0059
Epoch 10/10
311/311 [==============================] - 2s 5ms/step - loss: 0.0064
Mean Squared Error on Test Data: 0.006437881384044886
311/311 [==============================] - 2s 6ms/step


In [14]:
import plotly.graph_objs as go

# Create traces
trace0 = go.Scatter(
    x = list(range(len(predictions_original.reshape(-1,1)))),
    y = predictions_original.reshape(-1,1),
    mode = 'lines',
    name = 'Predictions'
)

trace1 = go.Scatter(
    x = list(range(len(y_test))),
    y = scaler.inverse_transform(y_test.reshape(-1,1)).reshape(-1),
    mode = 'lines',
    name = 'True'
)

data = [trace0, trace1]

# Edit the layout
layout = dict(title = 'Load Forecasting',
              xaxis = dict(title = 'Time'),
              yaxis = dict(title = 'Load'))

# Create the figure
fig = dict(data=data, layout=layout)

# Plot the figure
import plotly.express as px
fig = px.line(df, x=list(range(len(predictions_original))), y=predictions_original.reshape(-1), title='Predictions')
fig.add_scatter(x=list(range(len(y_test))), y=scaler.inverse_transform(y_test.reshape(-1,1)).reshape(-1), name='True')
fig.show()
layout = dict(title = 'Load Forecasting',
              xaxis = dict(title = 'Time'),
              yaxis = dict(title = 'Load'))

# Create the figure
fig = dict(data=data, layout=layout)

# Plot the figure
import plotly.express as px
fig = px.line(df, x=range(len(predictions_original)), y=predictions_original.reshape(-1), title='Predictions')
fig.add_scatter(x=range(len(y_test)), y=scaler.inverse_transform(y_test.reshape(-1,1)).reshape(-1), name='True')
fig.show()


ValueError: 
    Invalid value of type 'builtins.range' received for the 'x' property of scatter
        Received value: range(0, 9923)

    The 'x' property is an array that may be specified as a tuple,
    list, numpy array, or pandas Series